# 🍊 YOLOv8/11 Image Classification - SSD-First Colab Edition

**Cau hinh da duoc siet lai cho Colab + Drive:**
1. **SSD-first**: Train tren `/content` de nhanh hon, chi dong bo artifact quan trong len Drive.
2. **Resume an toan**: Neu co checkpoint tren Drive, runtime moi se copy ve SSD local roi tiep tuc train.
3. **Fail-fast mac dinh**: Khong am tham tao dummy dataset neu mount sai hoac thieu du lieu.
4. **Thuc dung hon cho Colab**: Cache mode, workers, preview va export ONNX duoc lam robust hon.

In [ ]:
# ============================================================
# Cell 0: Mount Google Drive + define local/drive paths
# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_SOURCE = '/content/drive/MyDrive/PBL5'
DRIVE_ROOT = '/content/drive/MyDrive/YOLO_PBL5_Classification'
DRIVE_SPLIT_DATASET = os.path.join(DRIVE_ROOT, 'dataset_split')
DRIVE_RUNS = os.path.join(DRIVE_ROOT, 'train_results')

LOCAL_ROOT = '/content/yolo_pbl5_cls'
LOCAL_DATASET = os.path.join(LOCAL_ROOT, 'dataset')
LOCAL_RUNS = os.path.join(LOCAL_ROOT, 'train_results')

for path in [DRIVE_ROOT, DRIVE_RUNS, LOCAL_ROOT, LOCAL_RUNS]:
    os.makedirs(path, exist_ok=True)

print(f'Drive source: {DRIVE_SOURCE}')
print(f'Drive root: {DRIVE_ROOT}')
print(f'Local dataset cache: {LOCAL_DATASET}')
print(f'Local runs: {LOCAL_RUNS}')

In [ ]:
# ============================================================
# Cell 1: Install libs + runtime tuning for Colab
# ============================================================
!pip install -q "ultralytics>=8.3.0,<9" split-folders onnx onnxsim

import os
import torch
import ultralytics
from ultralytics import YOLO

os.environ['YOLO_CONFIG_DIR'] = os.path.join(LOCAL_ROOT, '.config', 'ultralytics')
os.makedirs(os.environ['YOLO_CONFIG_DIR'], exist_ok=True)

def get_optimal_batch():
    if not torch.cuda.is_available():
        return 4
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram >= 14: return 64
    if vram >= 10: return 32
    return 16

BATCH_SIZE = get_optimal_batch()
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
CPU_COUNT = os.cpu_count() or 2
WORKERS = min(4, max(2, CPU_COUNT // 3))
CACHE_MODE = False
EPOCHS = 50
IMGSZ = 224
MAX_IMAGE_EDGE = 640
JPEG_QUALITY = 90
SAVE_PERIOD = 5
PATIENCE = 20

print(f'ultralytics version: {ultralytics.__version__}')
print(f'Device: {torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU"}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Workers: {WORKERS}')
print(f'Cache mode: {CACHE_MODE}')
print(f'Max local image edge: {MAX_IMAGE_EDGE}')
print(f'JPEG quality: {JPEG_QUALITY}')
if DEVICE == 'cpu':
    print('WARNING: Colab runtime is not using GPU. Training will be much slower.')

In [ ]:
# ============================================================
# Cell 2: Prepare classification dataset with fail-fast default
# ============================================================
import os
import shutil
from pathlib import Path

import numpy as np
import splitfolders
from PIL import Image, ImageOps, UnidentifiedImageError

CLASS_NAMES = ['cam', 'chanh', 'quyt']
ALLOW_DUMMY_DATASET = False
LOCAL_TRAIN_DATASET = os.path.join(LOCAL_ROOT, 'dataset_train_ready')
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

Image.MAX_IMAGE_PIXELS = None

def has_class_subdirs(root, classes):
    return os.path.isdir(root) and all(os.path.isdir(os.path.join(root, cls)) for cls in classes)

def has_split_dataset(root, classes):
    return all(has_class_subdirs(os.path.join(root, split_name), classes) for split_name in ['train', 'val'])

def print_disk_usage(label, path='/content'):
    usage = shutil.disk_usage(path)
    print(
        f"{label}: total={usage.total / 1e9:.2f} GB | used={usage.used / 1e9:.2f} GB | free={usage.free / 1e9:.2f} GB"
    )
    return usage

def remove_cache_artifacts(root):
    removed = 0
    root_path = Path(root)
    if not root_path.exists():
        return removed
    for path in root_path.rglob('*'):
        if path.is_file() and path.suffix.lower() in {'.npy', '.cache'}:
            path.unlink()
            removed += 1
    return removed

def summarize_split(root):
    summary = {}
    total = 0
    for split_name in ['train', 'val']:
        split_root = Path(root) / split_name
        summary[split_name] = {}
        for cls in CLASS_NAMES:
            count = 0
            cls_root = split_root / cls
            if cls_root.exists():
                count = sum(1 for p in cls_root.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS)
            summary[split_name][cls] = count
            total += count
    return summary, total

def print_summary(root, label):
    summary, total = summarize_split(root)
    print(f'{label} ({total} images):')
    for split_name in ['train', 'val']:
        counts = ', '.join(f"{cls}={summary[split_name][cls]}" for cls in CLASS_NAMES)
        print(f'  {split_name}: {counts}')

def make_train_ready_dataset(src_root, dst_root, max_edge, jpeg_quality):
    src_root = Path(src_root)
    dst_root = Path(dst_root)
    unreadable = []
    written = 0

    if dst_root.exists():
        shutil.rmtree(dst_root)
    dst_root.mkdir(parents=True, exist_ok=True)

    for split_name in ['train', 'val']:
        for cls in CLASS_NAMES:
            src_cls = src_root / split_name / cls
            dst_cls = dst_root / split_name / cls
            dst_cls.mkdir(parents=True, exist_ok=True)

            for src_path in sorted(src_cls.iterdir()):
                if not src_path.is_file() or src_path.suffix.lower() not in VALID_EXTENSIONS:
                    continue
                dst_path = dst_cls / f'{src_path.stem}.jpg'
                try:
                    with Image.open(src_path) as img:
                        img = ImageOps.exif_transpose(img).convert('RGB')
                        img.thumbnail((max_edge, max_edge), Image.Resampling.LANCZOS)
                        img.save(dst_path, format='JPEG', quality=jpeg_quality, optimize=True)
                        written += 1
                except (UnidentifiedImageError, OSError, ValueError) as exc:
                    unreadable.append(f'{src_path}: {exc}')

    if unreadable:
        preview = '\n'.join(unreadable[:10])
        raise RuntimeError(
            'Found unreadable images while preparing the local train-ready dataset. '
            f'Fix or remove them, then rerun the notebook.\n{preview}'
        )

    return written

print_disk_usage('Before cleanup')
removed_cache_files = remove_cache_artifacts(LOCAL_ROOT)
if removed_cache_files:
    print(f'Removed {removed_cache_files} stale cache artifact(s) from {LOCAL_ROOT}.')

for path in [LOCAL_DATASET, LOCAL_TRAIN_DATASET]:
    if os.path.exists(path):
        shutil.rmtree(path)

if not has_class_subdirs(DRIVE_SOURCE, CLASS_NAMES):
    if not ALLOW_DUMMY_DATASET:
        raise FileNotFoundError(
            f'Missing dataset in {DRIVE_SOURCE}. Expected class folders: {CLASS_NAMES}. '
            'Fix the Drive path or set ALLOW_DUMMY_DATASET=True only for notebook debugging.'
        )

    print('WARNING: dataset not found. Creating dummy dataset because ALLOW_DUMMY_DATASET=True.')
    for split_name in ['train', 'val']:
        for cls in CLASS_NAMES:
            cls_dir = os.path.join(LOCAL_TRAIN_DATASET, split_name, cls)
            os.makedirs(cls_dir, exist_ok=True)
            for idx in range(5):
                img_arr = np.random.randint(0, 255, (IMGSZ, IMGSZ, 3), dtype=np.uint8)
                Image.fromarray(img_arr).save(os.path.join(cls_dir, f'dummy_{idx}.jpg'))
else:
    if not has_split_dataset(DRIVE_SPLIT_DATASET, CLASS_NAMES):
        if os.path.exists(DRIVE_SPLIT_DATASET):
            shutil.rmtree(DRIVE_SPLIT_DATASET)
        print('Creating persistent train/val split on Drive...')
        splitfolders.ratio(DRIVE_SOURCE, output=DRIVE_SPLIT_DATASET, seed=42, ratio=(0.8, 0.2))
    else:
        print(f'Found existing split dataset on Drive: {DRIVE_SPLIT_DATASET}')

    print('Copying split dataset from Drive to Colab SSD...')
    shutil.copytree(DRIVE_SPLIT_DATASET, LOCAL_DATASET)
    print_summary(LOCAL_DATASET, 'Original local dataset')

    print('Building train-ready local dataset (resized/compressed JPEG)...')
    written_images = make_train_ready_dataset(
        src_root=LOCAL_DATASET,
        dst_root=LOCAL_TRAIN_DATASET,
        max_edge=MAX_IMAGE_EDGE,
        jpeg_quality=JPEG_QUALITY,
    )
    print(f'Prepared {written_images} image(s) for training.')

TRAIN_DATASET = LOCAL_TRAIN_DATASET
print_summary(TRAIN_DATASET, 'Train-ready dataset')
print_disk_usage('After dataset preparation')
print(f'Train dataset ready at: {TRAIN_DATASET}')

In [ ]:
# ============================================================
# Cell 3: Train on local SSD and sync checkpoints back to Drive
# ============================================================
import os
import shutil

RUN_NAME     = 'fruit_cls_v1'
RUN_ROOT_LOCAL = os.path.join(LOCAL_RUNS, RUN_NAME)
RUN_ROOT_DRIVE = os.path.join(DRIVE_RUNS, RUN_NAME)

LOCAL_LAST_CKPT = os.path.join(RUN_ROOT_LOCAL, 'weights', 'last.pt')

if os.path.exists(RUN_ROOT_LOCAL):
    shutil.rmtree(RUN_ROOT_LOCAL)
if os.path.exists(RUN_ROOT_DRIVE):
    print('Found previous run on Drive. Copying it back to local SSD for resume...')
    shutil.copytree(RUN_ROOT_DRIVE, RUN_ROOT_LOCAL, dirs_exist_ok=True)

RESUME = os.path.exists(LOCAL_LAST_CKPT)

model = YOLO(LOCAL_LAST_CKPT if RESUME else 'yolov8n-cls.pt')

def sync_run_to_drive(*_):
    if os.path.exists(RUN_ROOT_LOCAL):
        shutil.copytree(RUN_ROOT_LOCAL, RUN_ROOT_DRIVE, dirs_exist_ok=True)

model.add_callback('on_model_save', sync_run_to_drive)

print(f"Training mode: {'resume from checkpoint' if RESUME else 'fresh start'}")

results = model.train(
    data        = TRAIN_DATASET,
    epochs      = EPOCHS,
    imgsz       = IMGSZ,
    batch       = BATCH_SIZE,
    project     = LOCAL_RUNS,
    name        = RUN_NAME,
    resume      = RESUME,
    exist_ok    = True,
    cache       = CACHE_MODE,
    save_period = SAVE_PERIOD,
    patience    = PATIENCE,
    workers     = WORKERS,
    device      = DEVICE
)

sync_run_to_drive()
PROJECT_PATH = DRIVE_RUNS
print(f'Synced training artifacts to Drive: {RUN_ROOT_DRIVE}')

In [ ]:
# ============================================================
# Cell 4: Đánh giá hệ thống Metrics & Draw (Đảm bảo Robust)
# ============================================================
from IPython.display import Image, display
import os

BEST_MODEL_PATH = os.path.join(DRIVE_RUNS, RUN_NAME, 'weights', 'best.pt')
RESULTS_PNG     = os.path.join(DRIVE_RUNS, RUN_NAME, 'results.png')

if os.path.exists(BEST_MODEL_PATH):
    print('Running validation on the best checkpoint...')
    val_model = YOLO(BEST_MODEL_PATH)
    metrics = val_model.val(data=TRAIN_DATASET, device=DEVICE)
    
    if os.path.exists(RESULTS_PNG):
        print('Training curves:')
        display(Image(filename=RESULTS_PNG, width=800))
else:
    val_model = None
    print('ERROR: best.pt was not found on Drive.')

In [ ]:
# ============================================================
# Cell 5: Testing trực quan bằng ngữ cảnh Global Variables
# ============================================================
import glob
import os
import shutil
from IPython.display import Image, display
from ultralytics import YOLO

BEST_MODEL_PATH = os.path.join(DRIVE_RUNS, RUN_NAME, 'weights', 'best.pt')
if ('val_model' not in globals() or globals().get('val_model') is None) and os.path.exists(BEST_MODEL_PATH):
    val_model = YOLO(BEST_MODEL_PATH)

val_imgs = []
for ext in ['jpg', 'jpeg', 'png', 'webp']:
    val_imgs.extend(glob.glob(os.path.join(TRAIN_DATASET, 'val', '*', f'*.{ext}')))

if not val_imgs:
    print('WARNING: no validation images were found for preview.')
elif 'val_model' not in globals() or globals().get('val_model') is None:
    print('ERROR: validation model is not available.')
else:
    preview_dir = os.path.join(LOCAL_ROOT, 'predict_preview')
    if os.path.exists(preview_dir):
        shutil.rmtree(preview_dir)
    res = val_model.predict(source=val_imgs[0], save=True, project=LOCAL_ROOT, name='predict_preview')
    res_path = os.path.join(res[0].save_dir, os.path.basename(val_imgs[0]))

    if os.path.exists(res_path):
        display(Image(filename=res_path, width=400))
    else:
        print('WARNING: preview image was not written as expected.')

    names_map = res[0].names
    best_idx = res[0].probs.top1
    confidence = res[0].probs.top1conf.item()
    
    print(f"\nPrediction: {names_map[best_idx].upper()} (confidence: {confidence:.2%})")

In [ ]:
# ============================================================
# Cell 6: Đóng gói kết quả ra Binary Tối Thượng ONNX
# ============================================================
import os
import shutil
from ultralytics import YOLO

BEST_MODEL_PATH = os.path.join(DRIVE_RUNS, RUN_NAME, 'weights', 'best.pt')
if ('val_model' not in globals() or globals().get('val_model') is None) and os.path.exists(BEST_MODEL_PATH):
    val_model = YOLO(BEST_MODEL_PATH)

if 'val_model' not in globals() or globals().get('val_model') is None:
    print('ERROR: best.pt is unavailable, cannot export ONNX.')
else:
    print('Exporting ONNX...')
    try:
        onnx_path = val_model.export(format='onnx', imgsz=IMGSZ, simplify=True)
    except Exception as export_error:
        print(f'simplify=True failed: {export_error}')
        print('Retrying with simplify=False ...')
        onnx_path = val_model.export(format='onnx', imgsz=IMGSZ, simplify=False)
    
    drive_onnx_path = os.path.join(DRIVE_RUNS, RUN_NAME, 'weights', 'best.onnx')
    os.makedirs(os.path.dirname(drive_onnx_path), exist_ok=True)
    onnx_path = os.path.abspath(onnx_path)
    drive_onnx_path = os.path.abspath(drive_onnx_path)
    
    if onnx_path == drive_onnx_path:
        print(f'ONNX already exported to: {drive_onnx_path}')
    else:
        shutil.copy2(onnx_path, drive_onnx_path)
        print(f'ONNX exported to: {drive_onnx_path}')